# 🌱 PlantGuard AI - Dataset Exploratory Data Analysis (EDA)

This notebook performs comprehensive Exploratory Data Analysis (EDA) on the PlantVillage dataset.

### Objectives:
- Inspect dataset directory structure and class distribution.
- Analyze image resolutions, color channels, and file formats.
- Detect potential data corruption or missing samples.
- Visualize class balance and sample images per plant category.

In [1]:
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

print("Libraries imported successfully!")

Libraries imported successfully!


# Visualization Style

In [4]:
sns.set_theme(
    style="whitegrid",
    context="notebook"
)

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 120

print("✅ Visualization style configured")

✅ Visualization style configured


# Dataset Path

In [6]:
DATASET_DIR = Path(
    "../datasets/raw/plantvillage/PlantVillage"
)

print("Dataset path:")
print(DATASET_DIR.resolve())

print("\nDataset exists:", DATASET_DIR.exists())

Dataset path:
C:\Users\sarka\OneDrive\Desktop\plant_disease_detection_system\ML\datasets\raw\PlantVillage\PlantVillage

Dataset exists: True


# Dataset Folders/Classes

In [7]:
classes = sorted([
    folder.name
    for folder in DATASET_DIR.iterdir()
    if folder.is_dir()
])

print(f"Total classes: {len(classes)}")

print("\nClasses:")
for index, class_name in enumerate(classes, start=1):
    print(f"{index:02d}. {class_name}")

Total classes: 15

Classes:
01. Pepper__bell___Bacterial_spot
02. Pepper__bell___healthy
03. Potato___Early_blight
04. Potato___Late_blight
05. Potato___healthy
06. Tomato_Bacterial_spot
07. Tomato_Early_blight
08. Tomato_Late_blight
09. Tomato_Leaf_Mold
10. Tomato_Septoria_leaf_spot
11. Tomato_Spider_mites_Two_spotted_spider_mite
12. Tomato__Target_Spot
13. Tomato__Tomato_YellowLeaf__Curl_Virus
14. Tomato__Tomato_mosaic_virus
15. Tomato_healthy


# count images

In [10]:
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

class_counts = {}

for class_name in classes:

    class_dir = DATASET_DIR / class_name

    count = sum(
        1
        for file in class_dir.rglob("*")
        if file.is_file()
        and file.suffix.lower() in IMAGE_EXTENSIONS
    )

    class_counts[class_name] = count


class_counts_df = (
    pd.DataFrame(
        list(class_counts.items()),
        columns=["Class", "Image_Count"]
    )
    .sort_values(
        "Image_Count",
        ascending=False
    )
    .reset_index(drop=True)
)

class_counts_df

,Class,Image_Count
0,Tomato__Tomato_YellowLeaf__Curl_Virus,3208
1,Tomato_Bacterial_spot,2127
2,Tomato_Late_blight,1909
3,Tomato_Septoria_leaf_spot,1771
4,Tomato_Spider_mites_Two_spotted_spider_mite,1676
5,Tomato_healthy,1591
6,Pepper__bell___healthy,1478
7,Tomato__Target_Spot,1404
8,Potato___Early_blight,1000
9,Potato___Late_blight,1000


# basic dataset statictics

In [11]:
total_images = class_counts_df["Image_Count"].sum()
min_images = class_counts_df["Image_Count"].min()
max_images = class_counts_df["Image_Count"].max()
mean_images = class_counts_df["Image_Count"].mean()
median_images = class_counts_df["Image_Count"].median()

print("📊 DATASET STATISTICS")
print("-" * 40)

print(f"Number of classes : {len(classes)}")
print(f"Total images      : {total_images:,}")
print(f"Minimum/class     : {min_images:,}")
print(f"Maximum/class     : {max_images:,}")
print(f"Mean/class        : {mean_images:,.2f}")
print(f"Median/class      : {median_images:,.2f}")

📊 DATASET STATISTICS
----------------------------------------
Number of classes : 15
Total images      : 20,638
Minimum/class     : 152
Maximum/class     : 3,208
Mean/class        : 1,375.87
Median/class      : 1,404.00


## 1. Dataset Path Setup & Overview

In [ ]:
DATASET_DIR = Path("../datasets/raw/plantvillage")

if not DATASET_DIR.exists():
    # Fallback path if running notebook from root
    DATASET_DIR = Path("ml/datasets/raw/plantvillage")

print(f"Target dataset directory: {DATASET_DIR.resolve()}")
print(f"Directory exists: {DATASET_DIR.exists()}")

## 2. Class Distribution Analysis

In [ ]:
class_counts = {}
for class_folder in sorted(DATASET_DIR.glob("*")):
    if class_folder.is_dir():
        images = list(class_folder.glob("*.JPG")) + list(class_folder.glob("*.jpg")) + list(class_folder.glob("*.png"))
        class_counts[class_folder.name] = len(images)

df_classes = pd.DataFrame(list(class_counts.items()), columns=["Class", "Image Count"])
df_classes = df_classes.sort_values(by="Image Count", ascending=False).reset_index(drop=True)

print(f"Total Plant Classes: {len(df_classes)}")
print(f"Total Images: {df_classes['Image Count'].sum()}")
df_classes.head(16)

## 3. Class Balance Visualization

In [ ]:
plt.figure(figsize=(14, 8))
ax = sns.barplot(x="Image Count", y="Class", data=df_classes, palette="viridis")
plt.title("PlantVillage Class Sample Distribution", fontsize=16, fontweight="bold")
plt.xlabel("Number of Images", fontsize=12)
plt.ylabel("Disease / Plant Class", fontsize=12)
plt.tight_layout()
plt.show()

## 4. Image Attribute Inspection (Dimensions & Color Channels)

In [ ]:
shapes = []
modes = []

for class_folder in DATASET_DIR.glob("*"):
    if class_folder.is_dir():
        for img_path in list(class_folder.glob("*.JPG"))[:50]:
            try:
                with Image.open(img_path) as img:
                    shapes.append(img.size)
                    modes.append(img.mode)
            except Exception as e:
                print(f"Corrupt image found: {img_path} - Error: {e}")

unique_shapes = set(shapes)
unique_modes = set(modes)

print(f"Unique Image Dimensions (W, H): {unique_shapes}")
print(f"Unique Image Color Modes: {unique_modes}")